In [46]:
import zipfile
import json
import polars as pl
from typing import List
from pathlib import Path
import re
from dataclasses import dataclass, field
from enum import Enum

In [3]:
RAW_EARNINGS_PATH = Path("D:\earnings_calls\earnings_calls")
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

# Code to clean the earnings calls

In [47]:
class ChunkType(Enum):
    OPENING = "opening_statement"
    QA = "q_and_a"

@dataclass
class TranscriptChunk:
    call_id: str
    chunk_id: int
    chunk_type: ChunkType
    
    content: List[str] = field(default_factory=list) 
    speakers: List[str] = field(default_factory=list)
    
    question_text: List[str] = field(default_factory=list)
    question_speakers: List[str] = field(default_factory=list)
    answer_text: List[str] = field(default_factory=list)
    answer_speakers: List[str] = field(default_factory=list)

    def to_dict(self):
        if self.chunk_type == ChunkType.OPENING:
            final_q = "N/A (Opening Statement)"
            final_q_speakers = "N/A"
            final_a = "\n".join(self.content)
            final_a_speakers = ", ".join(sorted(set(self.speakers)))
        else:
            final_q = "\n".join(self.question_text)
            final_q_speakers = ", ".join(sorted(set(self.question_speakers)))
            final_a = "\n".join(self.answer_text)
            final_a_speakers = ", ".join(sorted(set(self.answer_speakers)))

        return {
            'id': self.call_id,
            'chunk_id': self.chunk_id,
            'chunk_type': self.chunk_type.value,
            'question': final_q,
            'question_speakers': final_q_speakers,
            'answer': final_a,
            'answer_speakers': final_a_speakers
        }

In [112]:
class EarningsCallTranscript:
    """
    Label the speaker and metadata for each earnings call.
    The class returns the identified characteristics for the Earnings Call
    """
    
    def __init__(self, call_id, raw_dict):
        self.call_id = call_id
        self.raw_dict = raw_dict
        
        # attributes
        self.cleaned_lines = []
        self.managers = set()
        self.start_QA = None
        self.final_dataframe = pl.DataFrame()
        
    def _clean(self):
        """
        we need to clean the names of the people in the call
        fill in cleaned_lines
        """
        temp_items = []
        
        seq_pattern = re.compile(r'\[(\d+)\]')
        name_clean_pattern = re.compile(r'\s*-\s*\[\d+\]|\[\d+\]|\,')

        for key, text in self.raw_dict.items():
            # get the seq to keep the order
            seq_match = seq_pattern.search(key)
            seq_num = int(seq_match.group(1)) if seq_match else 0
            
            # clean name
            clean_name = name_clean_pattern.sub('', key).strip()
            clean_text = text.strip()

            if not clean_text or "[_" in clean_text:
                continue

            # TODO Actually deal with this, it could be a reporter or someone else, not necessarily want to throw away
            if "unidentified" in clean_name.lower():
                continue
            
            temp_items.append({
                'seq': seq_num,
                'speaker': clean_name,
                'text': text.strip(),
                'is_operator': 'Operator' in clean_name
            })

        # populate cleaned lines
        self.cleaned_lines = sorted(temp_items, key=lambda x: x['seq'])
        return self
    
    # identify the managers
    def _identify_managers_and_split(self):
        """
        Scans opening statements to find manager names. Individuals who speak after operator opens the call &
        Before the operator intervenes again to open the questions
        """
        call_opened = False

        for line in self.cleaned_lines:
            if not call_opened and line['seq'] > 100:
                raise ValueError(f"Operator start not found in first 10 lines for {self.call_id}")
            
            # open the call the first time we find operator
            if line['is_operator'] and not call_opened:
                call_opened = True
                continue 

            # ends the opening statemenst the second time we find the operator
            if call_opened and line['is_operator']:
                if "press" in line['text'].lower() or "question" in line['text'].lower():
                    self.start_QA = line['seq'] # the line that starts the QA is when the operator first speaks
                    break 

            # managers are the people that talk between the opening and closing of teh clal
            if call_opened and not line['is_operator']:
                self.managers.add(line['speaker'])

    # get the Q&A
    def parse(self):
        """
        Split between Q&A and chunk each transcript
        """
        # clean
        self._clean()
        if not self.cleaned_lines:
            raise ValueError(f"Transcript {self.call_id} has not been cleaned or empty")

        # identify managers
        self._identify_managers_and_split()
        if self.start_QA is None:
            return []

        # chunking
        final_chunks = []

        # opening statements by managers
        opening_chunk = TranscriptChunk(
            call_id=self.call_id,
            chunk_id=1,
            chunk_type=ChunkType.OPENING
        )

        for line in self.cleaned_lines:
            if line['seq'] >= self.start_QA: 
                break
            
            if not line['is_operator']:
                if line['speaker'] not in self.managers:
                    raise ValueError(f"unrecognized speaker {line['speaker']} in {self.call_id} opening statements")
                else:
                    opening_chunk.content.append(f"{line['speaker']}: {line['text']}")
                    opening_chunk.speakers.append(line['speaker'])

        # add opening to chunks
        if opening_chunk.content:
            final_chunks.append(opening_chunk.to_dict())

        # Q&A
        if self.start_QA:
            current_id = 2
            qa_chunk = TranscriptChunk(call_id=self.call_id, chunk_id=current_id, chunk_type=ChunkType.QA)
            in_answer_mode = False
            
            # TODO: this is because i dont have the metadata on the speakers, i will at some point but for now deal with the managers that don't show up
            last_manager_text = ""

            for line in self.cleaned_lines:
                if line['seq'] <= self.start_QA: continue
                if line['is_operator']: continue

                name = line['speaker']
                text = line['text']
                full_text = f"{name}: {text}"

                # if not in manager check if the manager asks someone else to answer
                if name not in self.managers and in_answer_mode:
                    # get first name
                    first_name = name.split(' ')[0] if name else ""
                    name_match = first_name.lower() in last_manager_text.lower()
                    
                    is_short = len(last_manager_text) < 200

                    handoff_keywords = [
                        'respond', 'take', 'comment', 'add', 'elaborate', 
                        'thoughts', 'handle', 'perspective', 'view', 'go ahead', 
                        'detail', 'follow up', 'answer', 'correct'
                    ]
                    has_intent = any(kw in last_manager_text.lower() for kw in handoff_keywords)
                    
                    greeting_starters = ["good morning", "hi ", "hello", "thanks", "welcome"]
                    is_just_greeting = any(last_manager_text.lower().startswith(g) for g in greeting_starters) and len(last_manager_text) < 50

                    # response was short and said the name and has intent and its not just a greet
                    if is_short and name_match and has_intent and not is_just_greeting:
                        self.managers.add(name)
                        print(f"DEBUG: Promoted {name} to Manager list due to handoff.")

                # Q&A 
                if name in self.managers:
                    # ANSWER
                    qa_chunk.answer_text.append(full_text)
                    qa_chunk.answer_speakers.append(name)
                    in_answer_mode = True
                    
                    # keep in window text
                    last_manager_text = text 
                else:
                    # QUESTION
                    if in_answer_mode:
                        # Save previous chunk
                        if qa_chunk.question_text or qa_chunk.answer_text:
                            final_chunks.append(qa_chunk.to_dict())
                        
                        # Reset
                        current_id += 1
                        qa_chunk = TranscriptChunk(call_id=self.call_id, chunk_id=current_id, chunk_type=ChunkType.QA)
                        in_answer_mode = False
                        last_manager_text = "" # no last manager, empty
                    
                    qa_chunk.question_text.append(full_text)
                    qa_chunk.question_speakers.append(name)

            # Save hanging chunk
            if qa_chunk.question_text or qa_chunk.answer_text:
                final_chunks.append(qa_chunk.to_dict())

        return pl.DataFrame(final_chunks)

# Run over all the files from 2001 to 2020

In [118]:
def clean_yearly_file(filepath, year):
    """
    Clean each individual file and return a polar dataframe
    """
    final_df = pl.DataFrame()  

    with zipfile.ZipFile(filepath, 'r') as z:
        file_names = z.namelist() 
        with z.open(file_names[0]) as f:
            data = json.load(f)
            
            all_calls = []
            
            for call_id, raw_content in data.items():
                try:
                    processed_earnings_call = EarningsCallTranscript(call_id, raw_content)
                    df_chunk = processed_earnings_call.parse()                    
                    if df_chunk is not None and not df_chunk.is_empty():
                        all_calls.append(df_chunk)
                        
                except Exception as e:
                    print(f"Failed on Call ID {call_id}: {e}") #TODO Fix this
                    continue
            if all_calls:
                final_df = pl.concat(all_calls)
            else:
                print(f"No data processed for year {year}")
        
    return final_df

In [119]:
final_chunks = clean_yearly_file(RAW_EARNINGS_PATH / f'transcripts_year{year}.zip', year)

DEBUG: Promoted WADE R. FENN to Manager list due to handoff.
DEBUG: Promoted KEVIN FREELAND to Manager list due to handoff.
DEBUG: Promoted DAVID READERMAN to Manager list due to handoff.
Failed on Call ID 138533812907: 'list' object has no attribute 'is_empty'
Failed on Call ID 137390535658: unrecognized speaker SUSAN CARR in 137390535658 opening statements
Failed on Call ID 138661754310: 'list' object has no attribute 'is_empty'
Failed on Call ID 138926327181: 'list' object has no attribute 'is_empty'
Failed on Call ID 140120416692: 'list' object has no attribute 'is_empty'
Failed on Call ID 139940785762: 'list' object has no attribute 'is_empty'
DEBUG: Promoted JOHN MCGINTY to Manager list due to handoff.
Failed on Call ID 140378729693: unrecognized speaker Lisa Anselio in 140378729693 opening statements
Failed on Call ID 139304383889: unrecognized speaker Nick Rolley in 139304383889 opening statements
Failed on Call ID 139127835944: unrecognized speaker Gregg Swearingen in 13912783

In [120]:
final_chunks

id,chunk_id,chunk_type,question,question_speakers,answer,answer_speakers
str,i64,str,str,str,str,str
"""138864923344""",1,"""opening_statement""","""N/A (Opening Statement)""","""N/A""","""DARREN JACKSON: Thanks Dick, a…","""DARREN JACKSON, RICHARD M. SCH…"
"""138864923344""",2,"""q_and_a""","""""","""""","""RICHARD M. SCHULZE: Wade, do y…","""RICHARD M. SCHULZE, WADE R. FE…"
"""138864923344""",3,"""q_and_a""","""DAN WEWER: And just one other …","""DAN WEWER""","""WADE R. FENN: You are asking a…","""WADE R. FENN"""
"""138864923344""",4,"""q_and_a""","""DAN WEWER: Yes.""","""DAN WEWER""","""DARREN JACKSON: Dan, I can tak…","""DARREN JACKSON"""
"""138864923344""",5,"""q_and_a""","""DAN WEWER: Right. Thank you.""","""DAN WEWER""","""RICHARD M. SCHULZE: Dan, I'd l…","""RICHARD M. SCHULZE"""
…,…,…,…,…,…,…
"""138795630804""",50,"""q_and_a""","""JIM TIFFANY: Thanks and good m…","""JIM TIFFANY""","""THOMAS GOLISANO: We have not s…","""THOMAS GOLISANO"""
"""138795630804""",51,"""q_and_a""","""JIM TIFFANY: Okay. Great. And …","""JIM TIFFANY""","""THOMAS GOLISANO: Around 40% of…","""THOMAS GOLISANO"""
"""138795630804""",52,"""q_and_a""","""JIM TIFFANY: Excellent. Thanks…","""JIM TIFFANY, MARK MCCOHEN""","""THOMAS GOLISANO: We already we…","""THOMAS GOLISANO"""


In [125]:
# Inside the path there's a zip for every year, making a list of all the files inside the folder
years_list = range(2001,2021)

# every zip has two files, one's the earnings parsed and one is not parsed, let's use the parsed one
# so we don't have to do double work but I have no idea how it was parsed   
for year in years_list:
    filepath = RAW_EARNINGS_PATH / f'transcripts_year{year}.zip'
    print(filepath)
    df_year = clean_yearly_file(filepath, year)
    df_year = df_year.with_columns(
        pl.lit(f"{year}").cast(pl.Int64).alias('year')
    )
    df_year.write_parquet(DATA_DIR / 'processed' / f'cleaned_transcripts_{year}.parquet')

D:\earnings_calls\earnings_calls\transcripts_year2001.zip
DEBUG: Promoted WADE R. FENN to Manager list due to handoff.
DEBUG: Promoted KEVIN FREELAND to Manager list due to handoff.
DEBUG: Promoted DAVID READERMAN to Manager list due to handoff.
Failed on Call ID 138533812907: 'list' object has no attribute 'is_empty'
Failed on Call ID 137390535658: unrecognized speaker SUSAN CARR in 137390535658 opening statements
Failed on Call ID 138661754310: 'list' object has no attribute 'is_empty'
Failed on Call ID 138926327181: 'list' object has no attribute 'is_empty'
Failed on Call ID 140120416692: 'list' object has no attribute 'is_empty'
Failed on Call ID 139940785762: 'list' object has no attribute 'is_empty'
DEBUG: Promoted JOHN MCGINTY to Manager list due to handoff.
Failed on Call ID 140378729693: unrecognized speaker Lisa Anselio in 140378729693 opening statements
Failed on Call ID 139304383889: unrecognized speaker Nick Rolley in 139304383889 opening statements
Failed on Call ID 13912